# Convex MPC 시뮬레이션 — 시각화 노트북

이 노트북은 **계산하지 않습니다.** 시나리오를 고르고, `run_scenario()` 를 부르고, 결과를 그립니다.

## 규칙

1. **여기에 알고리즘을 쓰지 않는다.** 제어/동역학/계획 코드는 전부 `sim_main/*.py` 에 있고
   그쪽만 테스트로 보호된다. 노트북에 붙여 쓴 로직은 리뷰도 테스트도 받지 못하고,
   MuJoCo·실기로 이관할 때 한 줄도 따라가지 못한다.
2. **좋은 파라미터를 찾았으면 `scenarios.py` 에 등록한다.** 셀 안에서만 사는 설정은
   커널을 재시작하는 순간 존재하지 않았던 것이 된다. 재현할 수 없는 결과는 결과가 아니다.
3. **셀 하나 = 재실행 단위.** 무거운 실행(셀 2)과 그림(셀 3 이후)이 분리돼 있으므로,
   플롯을 고칠 때 시뮬레이션을 다시 돌리지 않는다.

## 왜 이렇게 바꿨나

이전 버전은 338 줄짜리 셀 **하나**에 파라미터·초기화·제어 루프·로깅·플로팅이 전부
들어 있었다. 같은 제어 루프가 `scenario_runner.py` 와 `mujoco_main_ref.py` 에도 복사돼
있어 총 3 벌이었고, Step 1 에서 고친 결함 1~9 는 그중 한 벌에만 반영됐다.
`config.py` 가 상수 전용이 된 1-1 이후로 이 노트북은 `from config import get_13d_state`
때문에 **import 조차 되지 않는 상태**였는데, 아무도 몰랐다.

> 사본은 조용히 썩는다. 사본이 있다는 사실 자체가 결함이다.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import numpy as np

from quadruped_mpc import config as cfg
from quadruped_mpc.experiments.scenarios import SCENARIOS
from quadruped_mpc.experiments.scenario_runner import run_scenario
from quadruped_mpc.viz.visualization import (
    animate_quadruped,
    plot_state_tracking,
    plot_force_and_contact,
    plot_foot_trajectory_3d,
    plot_leg_angles,
    plot_foot_comparison,
    plot_r_feet_wf_over_time,
    plot_swing_progress,
)

print("등록된 시나리오:", list(SCENARIOS))

In [ ]:
# ── 무엇을 돌릴지 고르는 셀 (여기서 시간이 걸린다) ──────────────────────
# (a) 기준선과 동일한 조건으로 재현: SCENARIOS 를 그대로 쓴다.
params = SCENARIOS["S3_yaw"]

# (b) 탐색 중이라면 직접 준다. 단 쓸 만한 값을 찾으면 반드시 scenarios.py 에
#     이름을 붙여 등록할 것 (규칙 2). 등록해야 기준선과 테스트가 붙는다.
# params = dict(gait_name="trotting", v_des_x=0.5, omega_z_deg_s=10.0, duration_s=4.0)

print("실행:", params)
h = run_scenario(**params)

dt_log = float(h["t"][1] - h["t"][0])   # 로깅 주기. 아래 모든 플롯이 이 값을 쓴다.

# ── 그림을 보기 전에 숫자부터 본다 ──────────────────────────────────────
# 눈으로 그래프를 보고 "괜찮아 보이네" 하는 판단이 가장 위험하다.
if not h["completed"]:
    print(f"🚨 발산 @ {h['diverged_at_s']:.3f}s — 아래 그림은 발산 직전까지만 유효하다")
else:
    z, rp = h["com_pos"][:, 2], np.abs(h["rpy"][:, :2])
    print(f"✅ 완주 {h['t'][-1] + dt_log:.2f}s, {len(h['t'])} 샘플")
    print(f"   높이            {z.min():.4f} ~ {z.max():.4f} m  (목표 {cfg.leg_length_straight / 2:.3f})")
    print(f"   roll/pitch 최대 {np.rad2deg(rp.max()):.2f} deg")
    print(f"   yaw 최종        {np.rad2deg(h['rpy'][-1, 2]):.2f} deg")
    print(f"   전진 거리       {h['com_pos'][-1, 0] - h['com_pos'][0, 0]:.3f} m")
    print(f"   측면 이탈 최대   {np.abs(h['com_pos'][:, 1]).max():.4f} m")
    print(f"   스윙 위상 최대   {float(h['max_s']):.6f}   (1.0 초과 = 스윙이 예정보다 길어졌다)")

In [ ]:
# 상태 추종 — 참조(점선) 대비 실제(실선)
plot_state_tracking(h["state_13"], h["state_ref_13"], dt=dt_log)

In [ ]:
# 지면반력과 접촉 스케줄
# 스윙 구간(접촉 0)에서 힘이 정확히 0 인지 여기서 눈으로 한 번 더 확인한다.
plot_force_and_contact(h["grf"], h["contact"], h["t"][-1] + dt_log)

In [ ]:
# 발 궤적 3D — 실제 발 위치 vs Raibert 목표 착지점
plot_foot_trajectory_3d(
    h["feet_W"],
    history_Sa=h["contact"],
    history_x=h["state_13"],
    history_p_feet_des_wf=h["feet_des_W"],
)

In [ ]:
# 보조 진단 플롯 — 필요할 때만 주석을 푼다.
plot_leg_angles(h["q"], dt_log)
plot_foot_comparison(h["feet_des_rel_W"], h["feet_rel_W"], dt=dt_log, leg_idx=0)
plot_r_feet_wf_over_time(h["feet_rel_W"], dt=dt_log)
plot_swing_progress(h["swing_phase"], dt=dt_log)

In [ ]:
# 3D 애니메이션 (가장 무겁다 — 마지막에 한 번만)
anim_html = animate_quadruped(
    h["state_13"],
    h["R_W_B"],
    h["feet_rel_W"],
    h["grf"],
    cfg.body_length,
    cfg.body_width,
    dt=dt_log,
    save=True,
    history_q=h["q"],
    link_lengths=(cfg.link_hip, cfg.link_upper, cfg.link_lower),
)
anim_html